<a href="https://colab.research.google.com/github/nathaniel-fruni/diplomova-praca/blob/main/data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Príprava dát

In [21]:
# importy
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import StratifiedGroupKFold
from IPython.display import display

pd.set_option("display.max_colwidth", 250)
SEED = 42

## Načítanie dát

In [38]:
ds = load_dataset("TUKE-KEMT/senti-sk")

train = ds["train"].to_pandas()
test = ds["test"].to_pandas()

df = pd.concat([train.assign(split="train"), test.assign(split="test")], ignore_index=True)
before = pd.crosstab(df["split"], df["label"], margins=True, margins_name="total").reindex(["train", "test", "total"])

## Čistenie dát

### Duplicitné záznamy, odkazy a prázdne texty

Z textov sa odstránia aj odkazy (URL), ktoré nenesú informáciu o sentimente. Komentáre, ktoré obsahovali iba odkaz, sa tým stanú prázdnymi a odstránia sa spolu s ostatnými prázdnymi textami.

In [39]:
cols = ["id", "label", "user", "created_at", "text"]
dup = df.duplicated(subset=cols)
empty = df["text"].str.strip() == ""

print(f"Úplne zhodné záznamy: {dup.sum()} | Prázdne texty: {empty.sum()}")
df = df[~dup & ~empty]

Úplne zhodné záznamy: 3 | Prázdne texty: 17


### Cudzojazyčné komentáre

Kandidáti sa vyberajú rovnako ako vo fáze pochopenia dát: detekcia anglického, nemeckého alebo maďarského jazyka s istotou aspoň 0,9 a text s aspoň štyrmi slovami. Po ručnej kontrole sa odstránia iba potvrdené cudzojazyčné komentáre.

In [40]:
!pip install -q lingua-language-detector

In [41]:
from lingua import Language, LanguageDetectorBuilder

detector = LanguageDetectorBuilder.from_languages(
    Language.SLOVAK, Language.CZECH, Language.ENGLISH, Language.GERMAN, Language.HUNGARIAN
).build()

lang = df["text"].map(lambda t: (l := detector.detect_language_of(t)) and l.name)
foreign = lang.index[lang.isin(["ENGLISH", "GERMAN", "HUNGARIAN"])]
conf = df.loc[foreign, "text"].map(lambda t: detector.compute_language_confidence_values(t)[0].value)
words = df["text"].str.split().str.len()

candidates = conf.index[conf >= 0.9]
print(len(candidates))

with pd.option_context("display.max_rows", 100):
    display(df.loc[candidates, ["label", "split", "text"]].assign(jazyk=lang[candidates]).sort_values("jazyk"))

90


,label,split,text,jazyk
23,negative,train,some1 hacked my account on aim now i have to make a new one,ENGLISH
9655,negative,train,Yeah! Exactly! A good case of study. Of course,ENGLISH
9658,negative,train,And did you understand what was written there this time???,ENGLISH
9660,negative,train,Yess! one timeeee! We have cheap testing area...,ENGLISH
9934,negative,train,https://www.wsj.com/articles/as-covid-sweeps-europe-one-country-tries-a-new-tactic-test-the-entire-population-11603462995,ENGLISH
10419,negative,train,https://m.facebook.com/story.php?story_fbid=1533875946796422&id=732167026967322&m_entstream_source=video_home&player_suborigin=entry_point&player_format=permalink&anchor_composer=false,ENGLISH
10424,negative,train,https://www.facebook.com/mydlojano/posts/3802670649766995?__cft__[0]=AZUm7QZ63EuGYFUrup5pYq1GgO-hdziyl-JNCZ-RsS23GvojAvgILFzRo3G5-lk-2HR6Gqvt8xlpXqFFSl_WSO7ePWgL8_Jc3PXppHBHHGSIMdYbT6Ry9yAplvQqaK2y1ssMQ7StRYUVp3dhtDNXCkZj&__tn__=%2CO%2CP-R,ENGLISH
10825,negative,train,What the heck are you doing? Do you actually know. Have you read the latest WHO and CDC feedback. You are selling your nations soul for your pride. You earn your money monthly whilst the nation starves. You temperamental MORON.https://fb.watch/1k...,ENGLISH
11687,negative,train,Once upon a time the God created Earth and Seas and then come Hero from universum named Igor Matovic (probably in some relationship with Darth Vader) And show over the whole world with Covid has began.,ENGLISH
11729,negative,train,Hopefully next time,ENGLISH


In [42]:
# indexy kandidátov, ktoré sú pri ručnej kontrole slovenské alebo české
keep = [13904, 13232, 5693, 5908, 11098, 12687, 6133]

foreign_rows = candidates.difference(keep)
print(f"Odstránené cudzojazyčné komentáre: {len(foreign_rows)}")
df = df.drop(foreign_rows)

Odstránené cudzojazyčné komentáre: 83


### Konfliktné označenia

Opakované texty s rozdielnymi labelmi sa hľadajú iba v trénovacej množine, aby sa pri rozhodovaní nevyužívali labely testovacej množiny. Testovacia množina ostáva nezmenená.

In [43]:
train = df[df["split"] == "train"]
conflict = train.index[train.groupby("text")["label"].transform("nunique") > 1]

print(f"Odstránené konfliktné záznamy z train: {len(conflict)}")
df = df.drop(conflict)

Odstránené konfliktné záznamy z train: 875


## Rozdelenie dát

### Validačná množina

Z trénovacej množiny sa vyčlení 10 % záznamov, stratifikovane podľa labelu a tak, aby sa rovnaký používateľ nevyskytoval v trénovacej aj validačnej množine.

In [44]:
train = df[df["split"] == "train"]
sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)
_, val_idx = next(sgkf.split(train, train["label"], groups=train["user"]))

df.loc[train.index[val_idx], "split"] = "validation"

### Podmnožiny pre VO2

Záznamy trénovacej množiny sa náhodne zoradia v rámci svojej triedy. Podmnožina s veľkosťou *p* obsahuje prvých *p* % záznamov z každej triedy, takže podmnožiny sú stratifikované a vnorené.

In [45]:
train = df[df["split"] == "train"].sample(frac=1, random_state=SEED)
by_label = train.groupby("label")
df.loc[train.index, "frac"] = (by_label.cumcount() + 1) / by_label["label"].transform("size")

sizes = [0.01, 0.05, 0.10, 0.25, 0.50, 1.00]
display(pd.DataFrame({f"{s:.0%}": df.loc[df["frac"] <= s, "label"].value_counts() for s in sizes}))

,1%,5%,10%,25%,50%,100%
label,,,,,,
negative,115,578,1157,2894,5788,11576
neutral,33,165,331,829,1658,3316
positive,14,70,140,352,704,1409


## Formát labelov

In [46]:
df["label_id"] = df["label"].map({"negative": 0, "neutral": 1, "positive": 2})
df["label_sk"] = df["label"].map({"negative": "negatívny", "neutral": "neutrálny", "positive": "pozitívny"})

## Výsledok prípravy

In [51]:
after = pd.crosstab(df["split"], df["label"], margins=True, margins_name="total").reindex(["train", "validation", "test", "total"])
display(pd.concat({"before": before, "after": after}))

label              negative  neutral  positive  total
       split                                         
before train          13513     3939      1696  19148
       test            3270      949       458   4677
       total          16783     4888      2154  23825
after  train          11576     3316      1409  16301
       validation      1347      399       144   1890
       test            3258      941       457   4656
       total          16181     4656      2010  22847

In [50]:
out_cols = ["text", "label", "label_id", "label_sk", "frac"]
for split in ["train", "validation", "test"]:
    df.loc[df["split"] == split, out_cols].to_parquet(f"{split}.parquet")

## Zhrnutie

Z datasetu boli odstránené 3 úplne zhodné záznamy, 17 prázdnych textov a 83 cudzojazyčných komentárov. Z trénovacej množiny bolo navyše odstránených 875 záznamov s konfliktnými labelmi, testovacia množina bola týmto krokom ponechaná bez zmeny. Trénovacia množina bola rozdelená na trénovaciu (16 301) a validačnú (1 890) časť, testovacia množina obsahuje 4 656 záznamov. Rozdelenie tried zostalo takmer nezmenené. Pripravené množiny sú uložené v súboroch `train.parquet`, `validation.parquet` a `test.parquet`.